# Project Lab 2: Key-Value Store and Route Planner

Notebook for **Chapter 16** of the *cs* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/cs/ch16.html

Tip: change a number, predict the output, then run the cell again.

## 16.1 Project: Mini Key-Value Store with Persistence

In [ ]:
import json, os

class KV:
    def __init__(self, path):
        self.path, self.index = path, {}
        open(path, "ab").close()          # make sure the file exists
        self._rebuild()
        self.f = open(path, "ab")

    def _rebuild(self):
        with open(self.path, "rb") as f:
            offset = 0
            for raw in f:
                try:
                    rec = json.loads(raw)
                except ValueError:        # half-written last line after a crash
                    break
                if rec.get("del"): self.index.pop(rec["k"], None)
                else:              self.index[rec["k"]] = offset
                offset += len(raw)
        if offset < os.path.getsize(self.path):
            with open(self.path, "r+b") as f: f.truncate(offset)

In [ ]:
    def _append(self, rec):
        pos = self.f.tell()
        self.f.write((json.dumps(rec) + "\n").encode()); self.f.flush(); os.fsync(self.f.fileno())
        return pos

    def put(self, k, v):  self.index[k] = self._append({"k": k, "v": v})
    def delete(self, k):
        if k in self.index: self._append({"k": k, "del": 1}); del self.index[k]

In [ ]:
    def get(self, k, default=None):
        if k not in self.index: return default
        with open(self.path, "rb") as f:
            f.seek(self.index[k]); return json.loads(f.readline())["v"]

In [ ]:
    def compact(self):
        tmp = self.path + ".tmp"; new_index = {}
        with open(tmp, "wb") as out:
            for k in self.index:
                pos = out.tell()
                out.write((json.dumps({"k": k, "v": self.get(k)}) + "\n").encode())
                new_index[k] = pos
            out.flush(); os.fsync(out.fileno())
        self.f.close(); os.replace(tmp, self.path)
        self.index = new_index; self.f = open(self.path, "ab")

    def size(self): return os.path.getsize(self.path)
    def close(self): self.f.close()

In [ ]:
import os, tempfile
from kvstore import KV
d = tempfile.mkdtemp(); p = os.path.join(d, "shop.db")
db = KV(p)
db.put("order:1", {"item": "pen", "qty": 3})
db.put("order:2", {"item": "book", "qty": 1})
db.put("order:1", {"item": "pen", "qty": 5})      # update
db.delete("order:2")
print("get order:1 ->", db.get("order:1"))
print("get order:2 ->", db.get("order:2"))
print("lines in file:", sum(1 for _ in open(p)), " bytes:", db.size())
db.close()
db = KV(p)                                        # restart
print("after restart, order:1 ->", db.get("order:1"))
for i in range(1000): db.put("hot", i)            # many overwrites of one key
before = db.size(); db.compact()
print("bytes before compact:", before, " after:", db.size(), " keys:", sorted(db.index))
print("hot ->", db.get("hot"))
db.close()
with open(p, "ab") as f: f.write(b'{"k": "order:9", "v": {"it')   # simulate a crash mid-write
db = KV(p); print("after torn write, keys:", sorted(db.index)); db.put("order:3", 1)
print("order:3 ->", db.get("order:3")); db.close()
print("reopen ->", sorted(KV(p).index))

In [ ]:
import os, tempfile
from kvstore import KV
p = os.path.join(tempfile.mkdtemp(), "t.db")
db = KV(p); db.put("a", 1); db.put("a", 2); db.delete("a"); db.put("b", 3); db.close()
db = KV(p)
assert db.get("a") is None and db.get("b") == 3
db.compact(); assert db.get("b") == 3 and sum(1 for _ in open(p)) == 1
print("kv tests ok")

## 16.2 Project: Graph-Based Route Planner

In [ ]:
import heapq
from collections import deque

ROADS = [("Depot","Market",4), ("Depot","Station",2), ("Station","Market",1),
         ("Market","School",5), ("Station","Hospital",8), ("School","Hospital",3),
         ("Hospital","Park",2), ("School","Park",6), ("Island","Lake",2)]

def build(roads):
    g = {}
    for a, b, w in roads:
        g.setdefault(a, []).append((b, w)); g.setdefault(b, []).append((a, w))
    return g

def bfs(g, start, goal):
    prev, q = {start: None}, deque([start])
    while q:
        u = q.popleft()
        if u == goal: break
        for v, _ in g[u]:
            if v not in prev: prev[v] = u; q.append(v)
    return path(prev, goal)

def dijkstra(g, start, goal):
    dist, prev, heap = {start: 0}, {start: None}, [(0, start)]
    while heap:
        d, u = heapq.heappop(heap)
        if d > dist[u]: continue          # stale entry
        if u == goal: break
        for v, w in g[u]:
            nd = d + w
            if nd < dist.get(v, float("inf")):
                dist[v] = nd; prev[v] = u; heapq.heappush(heap, (nd, v))
    p = path(prev, goal)
    return (p, dist[goal]) if p else (None, None)

def path(prev, goal):
    if goal not in prev: return None
    out = []
    while goal is not None: out.append(goal); goal = prev[goal]
    return out[::-1]

def minutes(g, p):
    return sum(w for a, b in zip(p, p[1:]) for n, w in g[a] if n == b)

In [ ]:
g = build(ROADS)
p = bfs(g, "Depot", "School");      print("BFS     :", " > ".join(p), "| stops:", len(p)-1, "| minutes:", minutes(g, p))
p, m = dijkstra(g, "Depot", "School"); print("Dijkstra:", " > ".join(p), "| stops:", len(p)-1, "| minutes:", m)
print("Depot -> Park :", dijkstra(g, "Depot", "Park"))
print("Depot -> Lake :", bfs(g, "Depot", "Lake"), dijkstra(g, "Depot", "Lake"))

In [ ]:
def all_paths(u, goal, seen):
    if u == goal: yield [u]; return
    for v, _ in g[u]:
        if v not in seen:
            for rest in all_paths(v, goal, seen | {v}): yield [u] + rest
best = min(minutes(g, q) for q in all_paths("Depot", "School", {"Depot"}))
print("brute-force best Depot -> School:", best)

## 16.3 Extension: Compaction, Checksums and a Grading Rubric

The store in 16.1 appends every write to a log and already has a `compact()` method and recovery from a half-written last line. What it cannot do is notice a record that was *changed* on disk (a flipped bit, a bad disk sector, a faulty copy), because a damaged line that still parses as JSON is accepted. Real log-structured stores (Bitcask, the write-ahead logs of LevelDB and Cassandra) protect every record with a checksum. The program below is a compact, self-contained version, with checksums and a crash-safe compaction, that you can paste into `kv2.py`.

In [ ]:
import json, os, tempfile, zlib

class KV2:
    def __init__(self, path):
        self.path, self.data = path, {}
        open(path, "ab").close()
        self._recover()
        self.f = open(path, "ab")

    def _line(self, rec):
        body = json.dumps(rec)
        return f"{zlib.crc32(body.encode()):08x}\t{body}\n".encode()

    def _recover(self):
        good = 0
        with open(self.path, "rb") as f:
            for raw in f:
                try:
                    crc, body = raw.decode().rstrip("\n").split("\t", 1)
                    ok = raw.endswith(b"\n") and int(crc, 16) == zlib.crc32(body.encode())
                    rec = json.loads(body) if ok else None
                except ValueError:
                    ok = False
                if not ok:
                    break
                if rec.get("del"): self.data.pop(rec["k"], None)
                else: self.data[rec["k"]] = rec["v"]
                good += len(raw)
        if good < os.path.getsize(self.path):
            with open(self.path, "r+b") as f: f.truncate(good)

    def _write(self, rec):
        self.f.write(self._line(rec)); self.f.flush(); os.fsync(self.f.fileno())

    def put(self, k, v): self._write({"k": k, "v": v}); self.data[k] = v
    def delete(self, k):
        if k in self.data: self._write({"k": k, "del": 1}); del self.data[k]

    def compact(self):
        tmp = self.path + ".tmp"
        with open(tmp, "wb") as t:
            for k, v in self.data.items(): t.write(self._line({"k": k, "v": v}))
            t.flush(); os.fsync(t.fileno())
        self.f.close()
        os.replace(tmp, self.path)
        self.f = open(self.path, "ab")

d = tempfile.mkdtemp(); p = os.path.join(d, "db.log")
db = KV2(p)
for i in range(1000): db.put("counter", i)
db.put("a", 1); db.put("b", 2); db.delete("b")
before = os.path.getsize(p)
db.compact()
after = os.path.getsize(p)
print("records before compaction: 1003, live keys:", len(db.data))
print("file shrank by more than 95 percent:", after < 0.05 * before, "(", after, "bytes left )")
db.f.close()
print("after reopen:", KV2(p).data)
# simulate a crash that left half a record at the end
with open(p, "ab") as f: f.write(b"deadbeef\t{\"k\": \"zz\", \"v\"")
print("after torn tail:", KV2(p).data)
# simulate a flipped bit inside a record
raw = open(p, "rb").read().replace(b'"v": 999', b'"v": 998')
open(p, "wb").write(raw)
print("after corrupted record:", KV2(p).data)

records before compaction: 1003, live keys: 2
file shrank by more than 95 percent: True ( 64 bytes left )
after reopen: {'counter': 999, 'a': 1}
after torn tail: {'counter': 999, 'a': 1}
after corrupted record: {}


## Chapter recap

## Practice set

Problem 16.1 (easy) You write `put("a",1)`, `put("a",2)`, `delete("a")`, `put("b",3)`. How many lines are in the log, and how many are alive after replay?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/cs/ch16.html